# NB00 - Bootstrap

**Run once, before anything else.** Sets up the Hugging Face dataset repo, uploads the shared library that every other notebook imports, and writes `config.json` (the single source of truth for the whole build).

### Prerequisites
- **Internet: ON** (Kaggle: Settings -> Internet -> On). Needed to reach Hugging Face.
- **GPU: not required** here. Leave it off to save quota.
- **HF token:** store a *write*-scoped token as a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets). On Colab/local it falls back to an environment variable or a prompt.
- **ImageNet terms:** visit `huggingface.co/datasets/ILSVRC/imagenet-1k` once on the account behind your token and accept the access conditions (NB01 needs it).

This notebook is idempotent - safe to re-run.

In [1]:
# Dependencies
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
pip("huggingface_hub>=0.23", "datasets", "pyarrow", "pillow")
print("deps installed")

deps installed


## 1. Set your repository id and load the token
Edit `REPO_ID` to your namespace. Keep it identical in every notebook.

In [2]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"   # <--- EDIT THIS
PRIVATE = False   # keep private until validated (NB10), then NB11 can flip it public

import os, json
from huggingface_hub import HfApi, whoami

# token: Kaggle secret -> env var -> prompt
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass
    return getpass.getpass("HF write token: ").strip()

TOKEN = load_token()
me = whoami(token=TOKEN)
print("authenticated as:", me.get("name"))
assert "/" in REPO_ID and not REPO_ID.startswith("your-username"), "Please edit REPO_ID first."

authenticated as: Shanmuk4622


In [3]:
# Create the dataset repo (idempotent)
api = HfApi(token=TOKEN)
api.create_repo(repo_id=REPO_ID, repo_type="dataset", private=PRIVATE, exist_ok=True)
print("repo ready:", REPO_ID, "(private)" if PRIVATE else "(public)")

repo ready: Shanmuk4622/ai-image-detection-dataset (public)


## 2. Write the shared library
`ai_detect_common.py` holds the canonical preprocess, the frozen schema, the robust uploader, and the verifier. Every notebook downloads **this exact file**, so real and AI images are guaranteed to be processed identically - that is what keeps the dataset leak-free.

In [4]:
%%writefile ai_detect_common.py
"""
ai_detect_common.py  --  shared library for the AI-image-detection dataset build.

This file is written and uploaded by NB00, then downloaded and imported *identically*
by every other notebook. That is deliberate: real images and AI images must pass
through the SAME preprocessing code, or the dataset leaks. Do not fork this logic
into individual notebooks.

PIPELINE_VERSION must match config.json. Bump it ONLY together with a full
regeneration of the dataset.
"""
from __future__ import annotations

import io
import os
import json
import time
import uuid
import getpass
import hashlib
import tempfile
import datetime
from collections import Counter

from PIL import Image, ImageOps
import pyarrow as pa
import pyarrow.parquet as pq

# Hugging Face hub is present on Kaggle/Colab; guard the import so this module can
# still be imported (e.g. for unit-testing the pure functions) without it.
try:
    from huggingface_hub import (
        HfApi, hf_hub_download, list_repo_files, CommitOperationAdd,
    )
    from huggingface_hub.utils import HfHubHTTPError
    _HF_AVAILABLE = True
except Exception:  # pragma: no cover
    _HF_AVAILABLE = False
    class HfHubHTTPError(Exception):
        pass

PIPELINE_VERSION = "1.1"
TARGET = 512
JPEG_QUALITY = 95

# ============================================================ IMAGING
def canonical_preprocess(img: "Image.Image") -> bytes:
    """The single canonical transform applied to EVERY image, real or AI.

    Returns final PNG bytes. NEVER branch on the label inside this function --
    real and AI must be byte-for-byte indistinguishable in their processing.
    """
    img = ImageOps.exif_transpose(img)              # 1. honour orientation BEFORE stripping
    img = img.convert("RGB")                         # 2. force 3-channel; drop alpha/ICC/bit-depth
    w, h = img.size                                  # 3. center-crop to square
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    img = img.crop((left, top, left + s, top + s))
    img = img.resize((TARGET, TARGET), Image.LANCZOS)  # 4. identical resample kernel
    img = Image.frombytes("RGB", img.size, img.tobytes())  # 5. strip ALL metadata, fast
    buf = io.BytesIO()                               # 6. JPEG equalisation: identical quant + 4:4:4
    img.save(buf, format="JPEG", quality=JPEG_QUALITY, subsampling=0)
    buf.seek(0)
    img = Image.open(buf).convert("RGB")
    img.load()
    out = io.BytesIO()                               # 7. lossless PNG container, identical encoder
    img.save(out, format="PNG", compress_level=6)
    return out.getvalue()


def decode_png(b: bytes) -> "Image.Image":
    im = Image.open(io.BytesIO(b))
    im.load()
    return im


def png_is_canonical(png_bytes: bytes):
    """Cheap structural check used by the verifiers. Returns (ok, reason)."""
    try:
        im = Image.open(io.BytesIO(png_bytes))
        im.load()
    except Exception as e:
        return False, f"undecodable: {e}"
    if im.format != "PNG":
        return False, f"format={im.format}"
    if im.mode != "RGB":
        return False, f"mode={im.mode}"
    if im.size != (TARGET, TARGET):
        return False, f"size={im.size}"
    if im.info.get("icc_profile"):
        return False, "has ICC profile"
    exif = getattr(im, "_getexif", lambda: None)()
    if exif:
        return False, "has EXIF"
    return True, "ok"


# ============================================================ SEEDS / HASHES / TIME
def make_seed(model: str, source_real_id: str) -> int:
    """Deterministic seed for (model, image). A re-run regenerates byte-identical
    output, so crashes / retries / double-runs cannot create divergent duplicates."""
    h = hashlib.sha256(f"{model}:{source_real_id}".encode("utf-8")).hexdigest()
    return int(h[:15], 16) % (2**31 - 1)   # safe range for numpy & torch generators


def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def now_utc() -> str:
    return datetime.datetime.now(datetime.timezone.utc).isoformat()


def get_device_dtype():
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda", torch.float16
        return "cpu", torch.float32
    except Exception:
        return "cpu", None


# ============================================================ FROZEN SCHEMAS
SCHEMA = pa.schema([
    ("image_id",         pa.string()),    # globally unique, prefixed: real_000001 / sdxl_000001
    ("source_real_id",   pa.string()),    # pairing key; == image_id for real rows
    ("label",            pa.int8()),       # 0 = real, 1 = ai
    ("generator",        pa.string()),     # real | sd15 | sdxl | flux_schnell | ...
    ("source_dataset",   pa.string()),     # coco | imagenet | <generator>
    ("prompt",           pa.string()),     # caption used (AI rows); null for real
    ("image",            pa.binary()),     # canonical PNG bytes
    ("width",            pa.int16()),
    ("height",           pa.int16()),
    ("orig_width",       pa.int32()),       # provenance ONLY -- never a training feature
    ("orig_height",      pa.int32()),
    ("gen_model_id",     pa.string()),
    ("gen_steps",        pa.int16()),
    ("gen_guidance",     pa.float32()),
    ("gen_native_res",   pa.int16()),
    ("seed",             pa.int64()),
    ("caption_model",    pa.string()),
    ("pipeline_version", pa.string()),
    ("sha256",           pa.string()),
    ("created_utc",      pa.string()),
])

CAPTION_SCHEMA = pa.schema([
    ("source_real_id", pa.string()),
    ("caption",        pa.string()),
    ("raw_caption",    pa.string()),
    ("caption_model",  pa.string()),
    ("caption_task",   pa.string()),
    ("n_tokens",       pa.int16()),
    ("created_utc",    pa.string()),
])


def empty_row() -> dict:
    return {f.name: None for f in SCHEMA}


# ============================================================ TOKEN LOADING
def load_hf_token() -> str:
    """Kaggle secret -> environment variable -> interactive prompt."""
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k):
            return os.environ[k].strip()
    return getpass.getpass("Enter your Hugging Face token (write scope): ").strip()


# ============================================================ ROBUST HF I/O
def robust_commit(api, repo_id, operations, msg, retries=6):
    """create_commit wrapped in exponential backoff for 429 / 5xx / transient errors."""
    delay = 5.0
    for attempt in range(1, retries + 1):
        try:
            api.create_commit(repo_id=repo_id, repo_type="dataset",
                              operations=operations, commit_message=msg)
            return True
        except HfHubHTTPError as e:
            code = getattr(getattr(e, "response", None), "status_code", None)
            transient = (code in (429, 500, 502, 503, 504)) or (code is None)
            if attempt == retries or not transient:
                raise
            print(f"  commit retry {attempt}/{retries} (HTTP {code}); sleeping {delay:.0f}s")
            time.sleep(delay)
            delay = min(delay * 2, 300)
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  commit retry {attempt}/{retries} ({type(e).__name__}); sleeping {delay:.0f}s")
            time.sleep(delay)
            delay = min(delay * 2, 300)
    return False


def read_config(repo_id, token) -> dict:
    p = hf_hub_download(repo_id, "config.json", repo_type="dataset", token=token)
    with open(p) as f:
        return json.load(f)


def list_shards(repo_id, prefix, token):
    files = list_repo_files(repo_id, repo_type="dataset", token=token)
    return sorted(f for f in files if f.startswith(prefix) and f.endswith(".parquet"))


def reconcile_ids(repo_id, prefix, token) -> set:
    """Authoritative resume set: source_real_ids already present in the shards.
    Reads ONLY the source_real_id column (no image decode) -- fast."""
    done = set()
    for f in list_shards(repo_id, prefix, token):
        try:
            local = hf_hub_download(repo_id, f, repo_type="dataset", token=token)
            tbl = pq.read_table(local, columns=["source_real_id"])
            done.update(tbl.column("source_real_id").to_pylist())
        except Exception as e:
            print(f"  WARN could not read {f}: {e}")
    return done


def count_rows(repo_id, prefix, token) -> int:
    """Authoritative count: rows in the parquet shards. progress.json is never trusted."""
    n = 0
    for f in list_shards(repo_id, prefix, token):
        local = hf_hub_download(repo_id, f, repo_type="dataset", token=token)
        n += pq.ParquetFile(local).metadata.num_rows
    return n


# ============================================================ BUFFERED SHARD WRITER
class ShardWriter:
    """Accumulates rows in memory and flushes a parquet shard to the repo on a
    time interval (default 20 min), when a row cap is hit, or on close().

    Shard filenames are unique per session, so sequential flushes -- and even two
    sessions writing the same generator -- never collide.
    """
    def __init__(self, api, repo_id, prefix, schema=None,
                 commit_interval=1200, max_rows=500, token=None):
        self.api = api
        self.repo_id = repo_id
        self.prefix = prefix.rstrip("/") + "/"
        self.tag = self.prefix.strip("/").split("/")[-1]
        self.schema = schema if schema is not None else SCHEMA
        self.commit_interval = commit_interval
        self.max_rows = max_rows
        self.token = token
        self.session = uuid.uuid4().hex[:8]
        self.seq = 0
        self.buf = []
        self.last_flush = time.time()
        self.total_committed = 0

    def add(self, row: dict):
        self.buf.append(row)
        if len(self.buf) >= self.max_rows or (time.time() - self.last_flush) >= self.commit_interval:
            self.flush()

    def maybe_flush(self):
        if self.buf and (time.time() - self.last_flush) >= self.commit_interval:
            self.flush()

    def flush(self):
        if not self.buf:
            return
        tbl = pa.Table.from_pylist(self.buf, schema=self.schema)
        fname = f"{self.prefix}{self.tag}-{self.session}-{self.seq:05d}.parquet"
        tmp = os.path.join(tempfile.gettempdir(), os.path.basename(fname))
        pq.write_table(tbl, tmp, compression="zstd")
        op = CommitOperationAdd(path_in_repo=fname, path_or_fileobj=tmp)
        robust_commit(self.api, self.repo_id, [op], f"add {len(self.buf)} rows -> {fname}")
        self.total_committed += len(self.buf)
        print(f"  committed {len(self.buf)} rows ({self.total_committed} this run) -> {fname}")
        self.seq += 1
        self.buf = []
        self.last_flush = time.time()
        try:
            os.remove(tmp)
        except OSError:
            pass

    def close(self):
        self.flush()


# ============================================================ VERIFIER (used by NB01)
def verify_real_stage(repo_id, token, config, sample=200):
    """End-of-NB01 verifier: confirms the real-image stage is correct.
    Prints a PASS/FAIL report and returns a bool."""
    import random as _r
    print("=" * 64)
    print("NB01 VERIFIER  --  real-image stage")
    print("=" * 64)
    problems, warns = [], []

    # 1. config + version agreement
    try:
        cfg = read_config(repo_id, token)
    except Exception as e:
        print("FAIL: cannot read config.json:", e)
        return False
    if cfg.get("pipeline_version") != PIPELINE_VERSION:
        problems.append(f"pipeline_version mismatch: config={cfg.get('pipeline_version')} "
                        f"module={PIPELINE_VERSION}")

    # 2. shards + light columns across ALL shards
    shards = list_shards(repo_id, "real/", token)
    print(f"real/ shards found: {len(shards)}")
    if not shards:
        print("FAIL: no real/ shards found")
        return False

    ids, srcs, shas, n_rows = [], [], [], 0
    for f in shards:
        local = hf_hub_download(repo_id, f, repo_type="dataset", token=token)
        t = pq.read_table(local, columns=["image_id", "source_dataset", "sha256",
                                          "label", "generator", "source_real_id",
                                          "width", "height", "pipeline_version"])
        n_rows += t.num_rows
        ids += t.column("image_id").to_pylist()
        srcs += t.column("source_dataset").to_pylist()
        shas += t.column("sha256").to_pylist()
        if set(t.column("label").to_pylist()) - {0}:
            problems.append(f"{f}: contains a non-zero label in the real stage")
        if set(t.column("generator").to_pylist()) - {"real"}:
            problems.append(f"{f}: contains generator != 'real'")
        if set(t.column("width").to_pylist()) - {TARGET} or set(t.column("height").to_pylist()) - {TARGET}:
            problems.append(f"{f}: width/height not all {TARGET}")
        if set(t.column("pipeline_version").to_pylist()) - {PIPELINE_VERSION}:
            problems.append(f"{f}: pipeline_version column disagrees with module")
        pair_bad = sum(1 for a, b in zip(t.column("image_id").to_pylist(),
                                         t.column("source_real_id").to_pylist()) if a != b)
        if pair_bad:
            problems.append(f"{f}: {pair_bad} rows where source_real_id != image_id")

    print(f"total real rows: {n_rows}")

    # 3. counts vs targets (per source)
    targets = config["real_sources"]
    exp_total = sum(targets.values())
    if n_rows != exp_total:
        warns.append(f"row count {n_rows} != target {exp_total} "
                     f"(fine if still in progress, or after NB09 dedup)")
    cc = Counter(srcs)
    for s, want in targets.items():
        got = cc.get(s, 0)
        print(f"  {s}: {got} / {want}")
        if got < want:
            warns.append(f"{s} under target: {got}/{want}")

    # 4. uniqueness
    dup_ids = [k for k, v in Counter(ids).items() if v > 1]
    if dup_ids:
        problems.append(f"{len(dup_ids)} duplicate image_id(s), e.g. {dup_ids[:3]}")
    dup_sha = [k for k, v in Counter(shas).items() if v > 1]
    if dup_sha:
        warns.append(f"{len(dup_sha)} duplicate image bytes (sha256) -- NB09 will dedup")

    # 5. sampled pixel / canonical checks (decode actual images)
    bad_canon, bad_sha, checked = 0, 0, 0
    sample_shards = shards if len(shards) <= 3 else _r.sample(shards, 3)
    per = max(1, sample // len(sample_shards))
    for f in sample_shards:
        local = hf_hub_download(repo_id, f, repo_type="dataset", token=token)
        t = pq.read_table(local, columns=["image", "sha256"])
        m = t.num_rows
        idxs = range(m) if m <= per else _r.sample(range(m), per)
        imgs, sh = t.column("image"), t.column("sha256")
        for i in idxs:
            b = imgs[i].as_py()
            checked += 1
            ok, why = png_is_canonical(b)
            if not ok:
                bad_canon += 1
            if sha256_bytes(b) != sh[i].as_py():
                bad_sha += 1
    print(f"sampled {checked} images: canonical_fail={bad_canon}, sha_mismatch={bad_sha}")
    if bad_canon:
        problems.append(f"{bad_canon}/{checked} sampled images not canonical (size/mode/EXIF/ICC)")
    if bad_sha:
        problems.append(f"{bad_sha}/{checked} sampled images have a sha256 mismatch")

    # verdict
    print("-" * 64)
    for w in warns:
        print("WARN:", w)
    if problems:
        print("\nRESULT: FAIL")
        for p in problems:
            print("  -", p)
        return False
    print("\nRESULT: PASS  --  real stage looks correct.")
    return True


Writing ai_detect_common.py


In [5]:
# Upload the library and sanity-import it
from huggingface_hub import CommitOperationAdd
api.upload_file(path_or_fileobj="ai_detect_common.py", path_in_repo="ai_detect_common.py",
                repo_id=REPO_ID, repo_type="dataset", commit_message="add/update shared library")
import importlib, ai_detect_common as C
importlib.reload(C)
print("library uploaded; PIPELINE_VERSION =", C.PIPELINE_VERSION)

library uploaded; PIPELINE_VERSION = 1.1


## 3. Write config.json
This locks every decision from the spec: targets, resolution, batch/commit cadence, captioner, and the per-model generation settings. The generators read this - do not hardcode these values elsewhere.

In [6]:
config = {
    "schema_version": "1.0",
    "pipeline_version": "1.1",
    "target_per_generator": 50000,
    "real_sources":    {"coco": 25000, "imagenet": 25000},
    "real_source_ids": {"coco": "HuggingFaceM4/COCO", "imagenet": "ILSVRC/imagenet-1k"},
    "target_resolution": 512,
    "jpeg_quality": 95,
    "batch_size": 500,                 # rows per shard / per commit cap
    "commit_interval_seconds": 1200,   # one batched commit every 20 minutes
    "caption_model": "microsoft/Florence-2-large",
    "caption_task": "<MORE_DETAILED_CAPTION>",
    "caption_max_tokens": 75,          # cap so CLIP-based models (SD1.5/SDXL) get the FULL prompt
    "split": {"train": 0.70, "val": 0.15, "test": 0.15, "seed": 1234},
    "generators": {
        "sd15":         {"model_id": "stable-diffusion-v1-5/stable-diffusion-v1-5", "pipeline": "sd",        "native": 768,  "steps": 30,         "guidance": 7.0},
        "sdxl":         {"model_id": "stabilityai/stable-diffusion-xl-base-1.0",    "pipeline": "sdxl",      "native": 1024, "steps": 30,         "guidance": 7.0},
        "flux_schnell": {"model_id": "black-forest-labs/FLUX.1-schnell",            "pipeline": "flux",      "native": 1024, "steps": 4,          "guidance": 0.0},
        "kandinsky22":  {"model_id": "kandinsky-community/kandinsky-2-2-decoder",   "pipeline": "kandinsky", "native": 1024, "steps": 50,         "guidance": 4.0},
        "pixart_sigma": {"model_id": "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS",      "pipeline": "pixart",    "native": 1024, "steps": 20,         "guidance": 4.5},
        "sd_cascade":   {"model_id": "stabilityai/stable-cascade", "prior_id": "stabilityai/stable-cascade-prior", "pipeline": "cascade", "native": 1024, "steps": [20, 10], "guidance": [4.0, 0.0]},
    },
}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)
api.upload_file(path_or_fileobj="config.json", path_in_repo="config.json",
                repo_id=REPO_ID, repo_type="dataset", commit_message="write config.json")
print("config.json written and uploaded.")
print(json.dumps(config, indent=2)[:600], "...")

config.json written and uploaded.
{
  "schema_version": "1.0",
  "pipeline_version": "1.1",
  "target_per_generator": 50000,
  "real_sources": {
    "coco": 25000,
    "imagenet": 25000
  },
  "real_source_ids": {
    "coco": "HuggingFaceM4/COCO",
    "imagenet": "ILSVRC/imagenet-1k"
  },
  "target_resolution": 512,
  "jpeg_quality": 95,
  "batch_size": 500,
  "commit_interval_seconds": 1200,
  "caption_model": "microsoft/Florence-2-large",
  "caption_task": "<MORE_DETAILED_CAPTION>",
  "caption_max_tokens": 75,
  "split": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15,
    "seed": 1234
  },
  "generators": {
    "sd15" ...


## 4. Environment check

In [7]:
import shutil
print("=" * 50, "\nENVIRONMENT\n", "=" * 50)
try:
    import torch
    print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU (fine for NB00/NB01)"))
    if torch.cuda.is_available():
        print("  VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
except Exception as e:
    print("torch not present (fine for NB00/NB01):", e)
try:
    import psutil
    print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 1))
except Exception:
    pass
print("disk free (GB):", round(shutil.disk_usage('.').free / 1e9, 1))
print("write access:", REPO_ID, "OK (repo created)")
print("\nNB00 complete. Next: run NB01 (real images), then NB02 (captions).")

ENVIRONMENT
torch: 2.10.0+cpu | CUDA: False | no GPU (fine for NB00/NB01)
RAM (GB): 33.7
disk free (GB): 20.9
write access: Shanmuk4622/ai-image-detection-dataset OK (repo created)

NB00 complete. Next: run NB01 (real images), then NB02 (captions).
